<a href="https://colab.research.google.com/github/DebDuttz/Training-D2/blob/main/FDP_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# We'll set this up properly in Part 2. This is just to build intuition.
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer("all-MiniLM-L6-v2")   # tiny, free, ~80MB

a = model.encode("I love my puppy")
b = model.encode("My dog is adorable")
c = model.encode("Please file your taxes")

print("puppy vs dog :", round(float(util.cos_sim(a, b)), 2))   # ~0.6  (similar!)
print("puppy vs tax :", round(float(util.cos_sim(a, c)), 2))   # ~0.05 (different)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

puppy vs dog : 0.72
puppy vs tax : 0.11


In [2]:
!pip install -q groq chromadb sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 87.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 132.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 86.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/

In [4]:
from google.colab import userdata
GROQ_API_KEY = userdata.get("GROQ_API_KEY")
print("✅ Key loaded" if GROQ_API_KEY else "❌ Missing key")

✅ Key loaded


In [5]:
documents = [
    "Acme Corp offers 24 days of paid annual leave to all full-time employees.",
    "Employees can work from home up to 3 days per week with manager approval.",
    "The office is located at 42 MG Road, Bangalore, and opens at 9:00 AM.",
    "Acme Corp reimburses internet bills up to 1000 rupees per month for remote staff.",
    "New employees are on probation for the first 6 months of employment.",
    "The annual company retreat happens every December in Goa.",
]

In [6]:
import chromadb

# In-memory database (resets when the notebook restarts — perfect for a lab)
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(name="acme_docs")

# Add our chunks. ChromaDB embeds them automatically. ✨
collection.add(
    documents=documents,
    ids=[f"doc_{i}" for i in range(len(documents))],   # each chunk needs a unique id
)

print(f"✅ Stored {collection.count()} chunks in the vector database.")

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:03<00:00, 27.4MiB/s]


✅ Stored 6 chunks in the vector database.


In [10]:
question = "For how many days an employee can work from home?"   # note: doc says "annual leave", not "holidays"

results = collection.query(query_texts=[question], n_results=2)

print("🔎 Top matching chunks:")
for chunk in results["documents"][0]:
    print(" -", chunk)

🔎 Top matching chunks:
 - Employees can work from home up to 3 days per week with manager approval.
 - Acme Corp offers 24 days of paid annual leave to all full-time employees.


In [15]:
from groq import Groq

groq_client = Groq(api_key=GROQ_API_KEY)
MODEL = "llama-3.3-70b-versatile"

def rag_answer(question: str, n_results: int = 2) -> str:
    # ① RETRIEVE the most relevant chunks
    results = collection.query(query_texts=[question], n_results=n_results)
    context = "\n".join(results["documents"][0])

    # ② AUGMENT: build a prompt that includes the context
    prompt = f"""Answer the question using ONLY the context below.
If the answer is not in the context, say "I don't have that information."

Context:
{context}

Question: {question}
Answer:"""

    # ③ GENERATE with Groq
    response = groq_client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,   # keep it factual, not creative
    )
    return response.choices[0].message.content

# 🧪 Try it
print(rag_answer("How many holidays do I get?"))

You get 24 days of paid annual leave.


In [16]:
print("📖 Acme Doc Bot ready! Ask about leave, WFH, office, etc. Type 'quit' to exit.\n")

while True:
    q = input("You: ")
    if q.strip().lower() in {"quit", "exit", "q"}:
        print("👋 Bye!")
        break
    if not q.strip():
        continue
    print("Bot:", rag_answer(q), "\n")

📖 Acme Doc Bot ready! Ask about leave, WFH, office, etc. Type 'quit' to exit.

You: leave
Bot: Acme Corp offers 24 days of paid annual leave to all full-time employees. 

You: where's the HO of the company
Bot: The office is located at 42 MG Road, Bangalore. 

You: what can i do there
Bot: I don't have that information. 



KeyboardInterrupt: Interrupted by user